You can download and run this notebook locally, or you can run it for free in a cloud environment using Colab or Sagemaker Studio Lab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kirbyju/TCIA_Notebooks/blob/main/TCIA_Aspera_CLI_Downloads.ipynb)

[![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github.com/kirbyju/TCIA_Notebooks/blob/main/TCIA_Aspera_CLI_Downloads.ipynb)

# Summary
Much of non-DICOM content in [The Cancer Imaging Archive (TCIA)](https://www.cancerimagingarchive.net/) is provided via links to IBM Aspera Faspex packages.  Aspera’s FASP protocol is designed to move data rapidly across networks with minimal disruption to other traffic.  Aspera’s Faspex application bundles data into packages that can be referenced via a web link (i.e., a URI).  When an Aspera Faspex link resolves in a browser, it presents a GUI that guides a user through the process of installing a browser extension and a local Aspera Connect client, if not already there, that moves the data using FASP between TCIA servers and the computer the browser is running on.

We frequently get requests from researchers for an option to download TCIA Faspex packages using a command line interface that bypasses the GUI.  While not a part of the standard Aspera distributions, the IBM Aspera developers have provided an open source tool (Apache 2.0 license) called [ascli (aspera-cli)](https://github.com/IBM/aspera-cli) that allows a client to download an Aspera Faspex package using its URI.  **This notebook is focused on demonstrating how to download TCIA data from Aspera packages via the command line on a linux system.**

**Note:** The performance of ascli appears to be significantly worse when using the free tier of Google Colab.  


# Setup

First we'll install some necessary prerequisite software.  First you need to install Ruby **>= 3.2**, then the aspera-cli gem, and afterward you can use that to install ascli.  The steps to [install Ruby](https://www.ruby-lang.org/en/downloads/) vary by operating system but the gem and ascli commands should be the same as the last two lines below.

In [ ]:
# To run shell commands in Colab
%%bash
set -e

# 1. Install dependencies
apt-get update -y
apt-get install -y gnupg2 curl build-essential

# 2. Import RVM signing keys
gpg2 --keyserver hkp://keyserver.ubuntu.com --recv-keys \
  409B6B1796C275462A1703113804BB82D39DC0E3 \
  7D2BAF1CF37B13E2069D6956105BD0E739499BDB || \
  (curl -sSL https://rvm.io/mpapis.asc | gpg2 --import - && \
   curl -sSL https://rvm.io/pkuczynski.asc | gpg2 --import -)

# 3. Install RVM
curl -sSL https://get.rvm.io | bash -s stable

# 4. Load RVM for this script
source /etc/profile.d/rvm.sh

# 5. Install Ruby 3.3.4 (>= 3.2 required by aspera-cli)
rvm install 3.3.4
rvm use 3.3.4 --default

# 6. Show Ruby version
ruby -v

# 7. Install aspera-cli
gem install aspera-cli -v 4.28.0

# 8. Show ascli version
ascli --version

# 9. Let ascli install its SDK and ascp client
ascli conf ascp install

If the ascli command above fails you can try this alternate 'local SDK installation' method described on https://github.com/IBM/aspera-cli:

```
curl -Lso sdk.zip https://ibm.biz/aspera_transfer_sdk
ascli config ascp install --sdk-url=file:///sdk.zip
```

The format is: file:///<path>, where <path> can be either a relative path (not starting with /), or an absolute path.

# Download an entire Aspera package

Now that the prerequisite installations are complete, you will be able to use the ascli **receive** command to transfer a TCIA Faspex package using its link by following these steps:
1.	Browse to the dataset landing page that describes the package,
2.	Find the “Download” button that one normally would use to download the package using the Faspex GUI, but do not click it,
3.	Right click that button icon and select “copy link address” or “copy link” or similar (depends on your browser) to extract the package URI into the clipboard or copy/paste buffer.

Replace the **url** parameter in the following cell with the link of the package you want to download.  

**Note:** If you don't change the package URL below it will only take a few seconds to download an example package (~100MB) from the [Cancer Moonshot Biobank - Gastroesophageal Cancer Collection (CMB-GEC)](https://doi.org/10.7937/E7KH-R486).


In [2]:
# To run shell commands in Colab
%%bash
set -e

# Load RVM + Ruby 3.3.4 in this shell -- each cell in Colab is a new shell
source /etc/profile.d/rvm.sh
rvm use 3.3.4 >/dev/null

# the actual ascli "receive" command with our package URL from above
ascli faspex5 packages receive --url='https://faspex.cancerimagingarchive.net/aspera/faspex/public/package?context=eyJyZXNvdXJjZSI6InBhY2thZ2VzIiwidHlwZSI6ImV4dGVybmFsX2Rvd25sb2FkX3BhY2thZ2UiLCJpZCI6Ijc2MiIsInBhc3Njb2RlIjoiZDg0NmZlM2Q5ZjZjNzliYjUxYWU2MWMzNjJkNmE1ODJmMTc0YmVkYSIsInBhY2thZ2VfaWQiOiI3NjIiLCJlbWFpbCI6ImhlbHBAY2FuY2VyaW1hZ2luZ2FyY2hpdmUubmV0In0='

Receiving package 762
+---------+---------+
| package | status  |
+---------+---------+
| 762     | success |
+---------+---------+


# List files in an Aspera package
If you'd like to get details about what's in a package before you start downloading you can use the following commands.

# Download individual files or directories from an Aspera package

In order to grab specific parts of a package you can use the **browse** parameter to look around inside the package.  Running the command without any path specified will show you the root folder of the package.  

Let's take a look at the [HER2 tumor ROIs](https://doi.org/10.7937/NVA3-N783) collection as an example.

If you'd like to recursively list every file and directory in the package and save the result to a CSV file you can use the following.  

Note that the formatting of the CSV gets a little weird when you use recursive due to inconsistencies in column headers for files versus directories.

In [4]:
# To run shell commands in Colab
%%bash
set -e

# Load RVM + Ruby 3.3.4 in this shell -- each cell in Colab is a new shell
source /etc/profile.d/rvm.sh
rvm use 3.3.4 >/dev/null

# example "browse" command
ascli --format=csv faspex5 packages browse --query=@json:'{"recursive":true}' --url="https://faspex.cancerimagingarchive.net/aspera/faspex/public/package?context=eyJyZXNvdXJjZSI6InBhY2thZ2VzIiwidHlwZSI6ImV4dGVybmFsX2Rvd25sb2FkX3BhY2thZ2UiLCJpZCI6IjczOSIsInBhc3Njb2RlIjoiNzEwNmUzNDFjMDY4MjljNjBkMmM0ZjcxYTBhMTE1ODcxNGIzZWNjNSIsInBhY2thZ2VfaWQiOiI3MzkiLCJlbWFpbCI6ImhlbHBAY2FuY2VyaW1hZ2luZ2FyY2hpdmUubmV0In0=" > package.csv

If you want to take a more interactive browsing approach you can use this to see the base directory of the package:

In [5]:
# To run shell commands in Colab
%%bash
set -e

# Load RVM + Ruby 3.3.4 in this shell -- each cell in Colab is a new shell
source /etc/profile.d/rvm.sh
rvm use 3.3.4 >/dev/null

# example "browse" command
ascli faspex5 packages browse --url="https://faspex.cancerimagingarchive.net/aspera/faspex/public/package?context=eyJyZXNvdXJjZSI6InBhY2thZ2VzIiwidHlwZSI6ImV4dGVybmFsX2Rvd25sb2FkX3BhY2thZ2UiLCJpZCI6IjczOSIsInBhc3Njb2RlIjoiNzEwNmUzNDFjMDY4MjljNjBkMmM0ZjcxYTBhMTE1ODcxNGIzZWNjNSIsInBhY2thZ2VfaWQiOiI3MzkiLCJlbWFpbCI6ImhlbHBAY2FuY2VyaW1hZ2luZ2FyY2hpdmUubmV0In0="

+-----------------------------------+----------------------------------+-----------+--------+----------------------+--------------------+
| path                              | basename                         | type      | size   | mtime                | permissions        |
+-----------------------------------+----------------------------------+-----------+--------+----------------------+--------------------+
| /HER2_tumor_ROIs_v3.sums          | HER2_tumor_ROIs_v3.sums          | file      | 62974  | 2024-03-23T21:07:20Z | view, edit, delete |
| /TCGA_BRCA_Filtered               | TCGA_BRCA_Filtered               | directory | <null> | <null>               | view, edit, delete |
| /Yale_HER2_cohort                 | Yale_HER2_cohort                 | directory | <null> | <null>               | view, edit, delete |
| /Yale_trastuzumab_response_cohort | Yale_trastuzumab_response_cohort | directory | <null> | <null>               | view, edit, delete |
+---------------------------------

Then, if you want to look at the contents of one of the directories, you just add the directory path to the end of this command.  Let's look at the 'Yale_HER2_cohort/Annotations' folder.

In [9]:
# To run shell commands in Colab
%%bash
set -e

# Load RVM + Ruby 3.3.4 in this shell -- each cell in Colab is a new shell
source /etc/profile.d/rvm.sh
rvm use 3.3.4 >/dev/null

# example "browse" command
ascli faspex5 packages browse --url="https://faspex.cancerimagingarchive.net/aspera/faspex/public/package?context=eyJyZXNvdXJjZSI6InBhY2thZ2VzIiwidHlwZSI6ImV4dGVybmFsX2Rvd25sb2FkX3BhY2thZ2UiLCJpZCI6IjczOSIsInBhc3Njb2RlIjoiNzEwNmUzNDFjMDY4MjljNjBkMmM0ZjcxYTBhMTE1ODcxNGIzZWNjNSIsInBhY2thZ2VfaWQiOiI3MzkiLCJlbWFpbCI6ImhlbHBAY2FuY2VyaW1hZ2luZ2FyY2hpdmUubmV0In0=" Yale_trastuzumab_response_cohort/Annotations

+------------------------------------------------------------+---------------+---------------+--------+----------------------+--------------------+--------------------------------------------------------------------------------------+-----------------+-------------+-------------+----------------------+--------------------+
| path                                                       | basename      | type          | size   | mtime                | permissions        | target.path                                                                          | target.basename | target.type | target.size | target.mtime         | target.permissions |
+------------------------------------------------------------+---------------+---------------+--------+----------------------+--------------------+--------------------------------------------------------------------------------------+-----------------+-------------+-------------+----------------------+--------------------+
| Yale_trastuzumab_respon

When you want to download a file, you can once again use the **recieve** command.  Let's pretend that we're interested in grabbing the entire annotation folder.  You can do that by simply appending the path of the directory to the end of the **receive** command.

In [7]:
# To run shell commands in Colab
%%bash
set -e

# Load RVM + Ruby 3.3.4 in this shell -- each cell in Colab is a new shell
source /etc/profile.d/rvm.sh
rvm use 3.3.4 >/dev/null

# example "receive" command
ascli faspex5 packages receive --url="https://faspex.cancerimagingarchive.net/aspera/faspex/public/package?context=eyJyZXNvdXJjZSI6InBhY2thZ2VzIiwidHlwZSI6ImV4dGVybmFsX2Rvd25sb2FkX3BhY2thZ2UiLCJpZCI6IjczOSIsInBhc3Njb2RlIjoiNzEwNmUzNDFjMDY4MjljNjBkMmM0ZjcxYTBhMTE1ODcxNGIzZWNjNSIsInBhY2thZ2VfaWQiOiI3MzkiLCJlbWFpbCI6ImhlbHBAY2FuY2VyaW1hZ2luZ2FyY2hpdmUubmV0In0=" Yale_trastuzumab_response_cohort/Annotations

Receiving package 739
+---------+---------+
| package | status  |
+---------+---------+
| 739     | success |
+---------+---------+


Now let's say that you're interested in obtaining just the first HER2 positive image and annotation file from the Yale HER2 cohort.  We can pull both files at the same time by appending their paths to the end of the previous **receive** command as shown here.

In [8]:
# To run shell commands in Colab
%%bash
set -e

# Load RVM + Ruby 3.3.4 in this shell -- each cell in Colab is a new shell
source /etc/profile.d/rvm.sh
rvm use 3.3.4 >/dev/null

# example "browse" command
ascli faspex5 packages receive --url="https://faspex.cancerimagingarchive.net/aspera/faspex/public/package?context=eyJyZXNvdXJjZSI6InBhY2thZ2VzIiwidHlwZSI6ImV4dGVybmFsX2Rvd25sb2FkX3BhY2thZ2UiLCJpZCI6IjczOSIsInBhc3Njb2RlIjoiNzEwNmUzNDFjMDY4MjljNjBkMmM0ZjcxYTBhMTE1ODcxNGIzZWNjNSIsInBhY2thZ2VfaWQiOiI3MzkiLCJlbWFpbCI6ImhlbHBAY2FuY2VyaW1hZ2luZ2FyY2hpdmUubmV0In0=" Yale_HER2_cohort/Annotations/Her2Pos_Case_01.xml Yale_HER2_cohort/SVS/Her2Pos_Case_01.svs

Receiving package 739
+---------+---------+
| package | status  |
+---------+---------+
| 739     | success |
+---------+---------+


# Download a participant cohort selected with tcia_utils

So far we've picked paths by hand. What if we already know which participants interest us? `tcia_utils` can find their indexed files and save a CSV that this notebook passes to Aspera.

Run **Setup** above first, then you can jump straight to this section. The earlier whole-package examples are independent. The Python package queries metadata and writes the file list; the separately installed Aspera CLI performs the transfers.

We'll start with open-access brain MR data and segmentation masks from [UCSF-PDGM](https://www.cancerimagingarchive.net/collection/ucsf-pdgm/) and [UPENN-GBM](https://www.cancerimagingarchive.net/collection/upenn-gbm/). Their images use NIfTI (`.nii` or `.nii.gz`). You can also load a participant cohort saved by the [TCIA queries notebook](https://colab.research.google.com/github/kirbyju/TCIA_Notebooks/blob/main/tcia_utils_v4/TCIA_Dataset_and_Participant_Queries.ipynb).

These are derived discovery metadata. The dataset descriptions and original supporting files remain the authoritative sources for cohort definitions and clinical values.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "tcia_utils[tables]==4.0.0a17", "nibabel>=5,<6"])

In [ ]:
from pathlib import Path
import json
import pandas as pd
from tcia_utils import TCIAClient, format_table

OUTPUT = Path("tcia-aspera-cohort")
OUTPUT.mkdir(exist_ok=True)
tcia = TCIAClient(base_url="https://tcia.duckdns.org", timeout=120, pin_snapshot=True)
COHORT_JSON = ""  # Optional path to a saved records-style participant cohort JSON.
MODALITY = "MR"
FILE_FORMATS = ["NIFTI"]
MAX_ITEMS = 2  # Files, not participants or bytes. Use None for the full selected file list.

Leave `COHORT_JSON` empty for the brain MR example. We'll find participants with public MR images and indexed segmentation files, then start with two participants. A participant appearing in both tables does not establish a particular image/mask pair.

To use a cohort you selected elsewhere, enter its JSON path above. Preserve `participant_key`, `dataset_type`, `short_title`, and `display_participant_id`; a participant ID alone is ambiguous across datasets. Change `MODALITY` and `FILE_FORMATS` for your question. Those participants must have exact indexed files in an open-access Aspera package; other download routes belong in their own workflows.

In [ ]:
if COHORT_JSON:
    selected_participants = pd.read_json(COHORT_JSON)
else:
    BRAIN_COLLECTIONS = ["UCSF-PDGM", "UPENN-GBM"]
    brain_candidates = tcia.search_participants(
        short_titles=BRAIN_COLLECTIONS,
        dataset_type="collection",
        modalities=["MR"],
        access_levels=["open"],
        file_formats=["NIFTI"],
        limit=500,
    ).to_dataframe(all_pages=True)
    brain_masks = tcia.search_public_assets(
        short_titles=BRAIN_COLLECTIONS,
        modalities=["MR"],
        file_formats=["NIFTI"],
        object_roles=["segmentation"],
        requires_annotations=True,
        limit=500,
    ).to_dataframe(all_pages=True)
    if brain_candidates.empty:
        brain_candidates = pd.DataFrame(
            columns=["short_title", "display_participant_id", "participant_key"]
        )
    if brain_masks.empty:
        brain_masks = pd.DataFrame(
            columns=["short_title", "subject_id", "asset_id", "file_name", "geometry_status"]
        )
    linked_masks = brain_masks.drop(
        columns=["participant_key", "display_participant_id"], errors="ignore"
    ).merge(
        brain_candidates[["short_title", "display_participant_id", "participant_key"]],
        left_on=["short_title", "subject_id"],
        right_on=["short_title", "display_participant_id"],
        how="left",
        validate="many_to_one",
        indicator="participant_link",
    )
    brain_cohort = linked_masks[linked_masks["participant_link"] == "both"].copy()
    print("Public MR participant records:", len(brain_candidates))
    print("Indexed segmentation files:", len(brain_masks))
    print(
        "Segmentation files without an exact inventory ID match:",
        linked_masks["participant_link"].eq("left_only").sum(),
    )
    display(
        brain_cohort.groupby("short_title").agg(
            participant_records=("participant_key", "nunique"),
            segmentation_files=("asset_id", "nunique"),
        )
    )
    display(brain_cohort[["short_title", "subject_id", "file_name", "geometry_status"]].head(10))
    brain_cohort.to_json(OUTPUT / "brain_mr_segmentation_inventory.json", orient="records", indent=2)

    selected_participants = brain_cohort[
        ["participant_key", "dataset_type", "short_title", "display_participant_id"]
    ].drop_duplicates().head(2).copy()

display(selected_participants)
selected_participants.to_json(OUTPUT / "selected_participants.json", orient="records", indent=2)

This tells us that a participant has both public MR data and segmentation files. Some masks label brain tissue and others label tumor regions; check the file descriptions for the labels you need. This table does **not** establish which individual MR volume each mask annotates. Use the dataset's file descriptions and naming documentation to select the corresponding source image.

Also, participant IDs do not always equal unique people. The UCSF-PDGM description explains that some original IDs were later recognized as follow-up scans of the same person. Use the collection's documented relationships when splitting training and test data; don't split related scans between them just because their filenames differ.

### What does the geometry information tell us?

Geometry describes the image grid: dimensions, voxel spacing, and orientation. `checked_grid_geometry` means a grid assessment was recorded. It does not prove that a mask matches a source image or that either is suitable for your model. `not_checked` means we don't have an assessment here.

Let's inspect one participant's source-image and segmentation records, including any detailed geometry information the service exposes.

In [ ]:
brain_files = pd.DataFrame()
if len(selected_participants):
    brain_example = selected_participants.iloc[0]
    brain_files = tcia.search_public_assets(
        short_titles=[brain_example["short_title"]],
        participant_id=brain_example["display_participant_id"],
        file_formats=FILE_FORMATS, limit=500,
    ).to_dataframe(all_pages=True)
    display(brain_files.reindex(columns=[
        "subject_id", "file_name", "object_role", "geometry_status",
        "geometry_assessment_method", "geometry_details_json",
    ]))
    brain_files.to_json(OUTPUT / "example_files.json", orient="records", indent=2)
else:
    print("No participants matched. Adjust the filters before exporting.")

### Save the selected file list

`export_aspera_manifest()` follows the exact participant-linked file paths in the index and matches them to current published package links. It does not guess folder names or download entire participant folders. This keeps indexed follow-up scans in scope as well.

The CSV has two columns: `packageUrl` and `packagePath`. It uses the same format as the Participant Explorer's Aspera file export, so an existing export can also be consumed by the transfer cell below.

`MAX_ITEMS=2` exports just two files across the cohort. Those first two files are not necessarily a matching source-image/mask pair, and two files can still be large. Inspect their names, sizes and the dataset documentation. Set `MAX_ITEMS=None` and rerun this cell to export all indexed matching files for the selected participants.

In [ ]:
aspera_manifest = tcia.export_aspera_manifest(
    selected_participants, OUTPUT / "tcia_aspera_files.csv",
    modality=MODALITY, file_formats=FILE_FORMATS,
    max_items=MAX_ITEMS, overwrite=True,
)
display(format_table(pd.DataFrame(aspera_manifest.rows), columns=[
    "short_title", "display_participant_id", "file_name", "size_bytes", "packagePath",
]))
display(pd.DataFrame(aspera_manifest.coverage))
print("Matching indexed files:", aspera_manifest.scope["matching_files"])
print("Files in this manifest:", aspera_manifest.scope["exported_files"])
print("Manifest:", aspera_manifest.path)
(OUTPUT / "aspera_selection.json").write_text(aspera_manifest.to_json(indent=2))

### Download the files in the manifest

This small adapter reads the CSV, groups files by package and passes the exact paths as separate arguments to `ascli`. It never turns a missing path into a whole-package download. The adapter lives here because it depends on the separate Aspera installation, rather than being part of `tcia_utils`.

The command uses the Ruby environment created in **Setup**. If you installed `ascli` another way, set `USE_NOTEBOOK_RVM=False`. The setup pins [ascli 4.28.0](https://github.com/IBM/aspera-cli/releases/tag/v4.28.0), which includes fixes for receiving lists of files through public package links.

You can set `MANIFEST_FILE` to an existing `tcia_aspera_files.csv` exported from the Participant Explorer instead. Review its rows before running the next cell.

In [ ]:
import csv
import subprocess
from collections import defaultdict
from pathlib import PurePosixPath
from urllib.parse import urlsplit, parse_qs

USE_NOTEBOOK_RVM = True
MANIFEST_FILE = OUTPUT / "tcia_aspera_files.csv"
DOWNLOAD_DIRECTORY = OUTPUT / "images"

def receive_aspera_manifest(manifest_file, destination, *, use_notebook_rvm=True):
    with Path(manifest_file).open(newline="", encoding="utf-8-sig") as source:
        reader = csv.DictReader(source)
        if reader.fieldnames != ["packageUrl", "packagePath"]:
            raise ValueError("Expected packageUrl,packagePath CSV columns.")
        rows = list(reader)
    if not rows:
        raise ValueError("The manifest contains no files; nothing will be downloaded.")
    groups = defaultdict(set)
    for row in rows:
        url, path = row["packageUrl"], row["packagePath"]
        parsed = urlsplit(url)
        if (parsed.scheme != "https" or parsed.hostname != "faspex.cancerimagingarchive.net"
                or parsed.username or parsed.password or parsed.port not in {None, 443}
                or parsed.fragment or parsed.path.rstrip("/") not in
                {"/aspera/faspex", "/aspera/faspex/public/package"}
                or set(parse_qs(parsed.query)) != {"context"}
                or not path or path.startswith(("-", "@")) or path.endswith("/")
                or ".." in PurePosixPath(path).parts or "\\" in path
                or any(ord(c) < 32 for c in url + path)):
            raise ValueError("The manifest contains an invalid public package/file route.")
        groups[url].add(path)
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    for number, (url, paths) in enumerate(groups.items(), start=1):
        ordered = sorted(paths)
        # Short batches keep large cohort exports within command-line limits.
        for offset in range(0, len(ordered), 50):
            batch = ordered[offset:offset + 50]
            print(f"Package {number}/{len(groups)}: transferring {len(batch)} selected files", flush=True)
            command = ["ascli", "faspex5", "packages", "receive",
                       "--url=" + url, "--to-folder=" + str(destination), *batch]
            if use_notebook_rvm:
                command = ["bash", "-lc",
                    'source /etc/profile.d/rvm.sh && rvm use 3.3.4 >/dev/null && exec "$@"',
                    "aspera-cohort", *command]
            # Stream Aspera progress, while keeping package URLs out of exception messages.
            if subprocess.call(command) != 0:
                raise RuntimeError(f"Aspera package {number} failed. Review its transfer output.")
    print("Aspera finished the selected transfers. Check its transfer summary for details.")

receive_aspera_manifest(MANIFEST_FILE, DOWNLOAD_DIRECTORY, use_notebook_rvm=USE_NOTEBOOK_RVM)

Once the files are on your computer, this optional check reads their NIfTI headers. Set the two paths to a **documented matching pair**. We'll compare grid shape, voxel sizes, orientation, and the affine matrix (which places the grid in physical space). A matching grid is a useful first check, not a replacement for visually checking the overlay or verifying label meanings from the dataset documentation.

In [ ]:
SOURCE_NIFTI = ""  # A downloaded source image, for example /path/to/image.nii.gz
MASK_NIFTI = ""  # Its documented matching mask.

if SOURCE_NIFTI and MASK_NIFTI:
    import nibabel as nib
    import numpy as np

    image, mask = nib.load(SOURCE_NIFTI), nib.load(MASK_NIFTI)
    display(
        pd.DataFrame(
            [
                {
                    "file": Path(path).name,
                    "shape": obj.shape,
                    "voxel_sizes": obj.header.get_zooms(),
                    "spatial_units": obj.header.get_xyzt_units()[0],
                    "orientation": nib.aff2axcodes(obj.affine),
                }
                for path, obj in [(SOURCE_NIFTI, image), (MASK_NIFTI, mask)]
            ]
        )
    )
    same_grid = (
        image.shape[:3] == mask.shape[:3]
        and image.header.get_xyzt_units()[0] != "unknown"
        and image.header.get_xyzt_units()[0] == mask.header.get_xyzt_units()[0]
        and np.allclose(image.affine, mask.affine, atol=1e-5)
    )
    print("Same spatial grid:", same_grid)
    print("Image affine:\n", image.affine)
    print("Mask affine:\n", mask.affine)
else:
    print(
        "Optional local-file check skipped. Fill in both paths after downloading a matching pair."
    )

In [ ]:
tcia.close()
print("Saved the participant cohort, exact file manifest and metadata selection in", OUTPUT)

# Acknowledgements
TCIA is funded by the [Cancer Imaging Program (CIP)](https://imaging.cancer.gov/), a part of the United States [National Cancer Institute (NCI)](https://www.cancer.gov/).  It is managed by the [Frederick National Laboratory for Cancer Research (FNLCR)](https://frederick.cancer.gov/) and hosted by the [University of Arkansas for Medical Sciences (UAMS)](https://www.uams.edu/)

This notebook was created by [Justin Kirby](https://www.linkedin.com/in/justinkirby82/), [Quasar Jarosz](https://www.linkedin.com/in/quasarjarosz/) and [Lawrence Tarbox](https://www.linkedin.com/in/lawrence-tarbox-088335/). Also, big thanks to [@VolodymyrChapman](https://github.com/VolodymyrChapman) for pointing out we can download individual files/directories from a package and to [Laurent Martin](https://www.linkedin.com/in/laurentmartinjp/) for assisting with a variety of our ASCLI questions!

If you leverage this notebook or any TCIA datasets in your work, please be sure to comply with the [TCIA Data Usage Policy](https://wiki.cancerimagingarchive.net/x/c4hF). In particular, make sure to cite the DOI(s) for the specific TCIA datasets you used in addition to the following paper!

## TCIA Citation

Clark, K., Vendt, B., Smith, K., Freymann, J., Kirby, J., Koppel, P., Moore, S., Phillips, S., Maffitt, D., Pringle, M., Tarbox, L., & Prior, F. (2013). The Cancer Imaging Archive (TCIA): Maintaining and Operating a Public Information Repository. Journal of Digital Imaging, 26(6), 1045–1057. https://doi.org/10.1007/s10278-013-9622-7